In [ ]:
%load_ext autoreload
%autoreload 2
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Anchor to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet, downsample_block_mean, fit_dtmc, _find_optimal_lattice_subset
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.utils.plotting import plot_markov_matrix
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import build_approach, AugmentedValueControl, AugmentedPolicyControl, AugmentedFCLockedControl

In [ ]:
class LinearGridBenchmarker(VoyageBenchmarker):
    def _get_or_compute_models(self, train_days: list, solver_cls, horizon_length: int):
        total_offline_time = 0.0
        
        # We change the hash so the Vault doesn't mix up Smart vs Linear grids
        markov_hash = self.vault.generate_markov_hash(train_days, self.config) + "_linear_fixed"
        loaded_mc = self.vault.load_markov_model(markov_hash)
        
        if loaded_mc is not None:
            mc_model, mc_time = loaded_mc
        else:
            train_t, train_pd = [], []
            for d in train_days:
                data = self.fleet_cache[d]
                t_off = train_t[-1][-1] if train_t else 0.0
                train_t.append(data['t'] + t_off)
                train_pd.append(data['Pd'])
                
            t_concat = np.concatenate(train_t)
            pd_concat = np.concatenate(train_pd)
            ds_train = downsample_block_mean(t_concat, pd_concat, self.config.Dt, align='t0')
            
            start_t = time.perf_counter()
            
            # --- THE CORRECTED LINEAR GRID OVERRIDE ---
            dP = self.config.dP
            max_observed = ds_train['Pd'].max()
            
            # 1. Create uniform boundary bins across the data domain
            ideal_edges = np.linspace(0.0, max_observed, self.config.N_Pd + 1)
            
            # 2. Extract the mathematical centers of these uniform bins
            ideal_levels = (ideal_edges[:-1] + ideal_edges[1:]) / 2.0
            
            # 3. Define the absolute physical dP lattice 
            max_lattice_val = max(max_observed + dP, self.config.N_Pd * dP)
            full_lattice = np.arange(0.0, max_lattice_val + (dP / 2.0), dP)
            
            # 4. Snap the ideal linear centers to the nearest dP multiples
            linear_levels_snapped = _find_optimal_lattice_subset(ideal_levels, full_lattice)
            
            # 5. Derive practical boundaries (edges) halfway between snapped nodes
            linear_edges = np.zeros(self.config.N_Pd + 1)
            linear_edges[0] = -1e-3  
            linear_edges[-1] = max_lattice_val + dP 
            for idx in range(1, self.config.N_Pd):
                linear_edges[idx] = (linear_levels_snapped[idx-1] + linear_levels_snapped[idx]) / 2.0
            
            mc_model = fit_dtmc(ds_train['Pd'], self.config.N_Pd, self.config.alpha_mc, 
                                manual_edges=linear_edges, manual_levels=linear_levels_snapped)
            
            mc_time = time.perf_counter() - start_t
            mc_model['Delta'] = self.config.Dt  
            self.vault.save_markov_model(markov_hash, mc_model, train_days, offline_time=mc_time)
            
        total_offline_time += mc_time
        
        # --- BELLMAN SOLVER ---
        raw_solution = None
        if solver_cls is not None:
            sdp_hash = self.vault.generate_sdp_hash(markov_hash, solver_cls.__name__, horizon_length, self.config) + "_linear_fixed"
            loaded_sdp = self.vault.load_sdp_model(sdp_hash)
            if loaded_sdp is not None:
                raw_solution, sdp_time = loaded_sdp
            else:
                solver = solver_cls(self.config, mc_model)
                start_t = time.perf_counter()
                raw_solution = solver.compute_solution(horizon_length)
                sdp_time = time.perf_counter() - start_t
                self.vault.save_sdp_model(sdp_hash, markov_hash, solver_cls.__name__, raw_solution, offline_time=sdp_time)
            total_offline_time += sdp_time
            
        return mc_model, raw_solution, total_offline_time

In [ ]:
env = EnvConfig()
fleet_data = load_and_cache_entire_fleet(env)

# Validation exclusion
exclude_days = [1, 2, 3]

# [LOCK IN YOUR OPTIMAL GRID]
# Substitute these with the absolute best values you found from Notebooks 1, 2 & 3.
OPTIMAL_DP = 150.0
OPTIMAL_DT = 300
N_PD = 6 # Locked low to ensure dense transition matrices
N_PACK = 4

# Shared configuration for both
config = SimConfig(
    dP=OPTIMAL_DP,
    Dt=OPTIMAL_DT, 
    N_Pd=N_PD, 
    use_smart_grid=True, # Both use the exact same integer lattice mechanics
    n_pack=N_PACK,
    apply_terminal_n_cost=False,
    apply_terminal_soc_cost=True,
    alpha_fc=4
)

# Factories
eval_factory = build_approach(
    controller_cls=AugmentedFCLockedControl, 
    plant_cls=AugmentedHybridPlant, 
    solver_cls=AugmentedHybridSDPSolver, 
    is_macro=False
)

In [ ]:
# Before simulating, let's extract and plot the Markov Matrices to see the difference
train_days = [d for d in fleet_data.keys() if d not in exclude_days and d != 9] # Use a sample training set

# 1. Standard Benchmarker (Quantile/Smart Grid)
benchmarker_smart = VoyageBenchmarker(fleet_data, env, config, exclude_days)
mc_smart, _, _ = benchmarker_smart._get_or_compute_models(train_days, AugmentedHybridSDPSolver, 1)

# 2. Linear Grid Benchmarker
benchmarker_linear = LinearGridBenchmarker(fleet_data, env, config, exclude_days)
mc_linear, _, _ = benchmarker_linear._get_or_compute_models(train_days, AugmentedHybridSDPSolver, 1)

# Plotting side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_markov_matrix(mc_linear, title="Naive Linear Grid Topology", ax=axes[0])
plot_markov_matrix(mc_smart, title="Quantile-Clustered Topology (Smart)", ax=axes[1])

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/markov_topology_comparison.png', dpi=300)
plt.show()

print("--- LATTICE ASSIGNMENT AUDIT ---")
print(f"Linear Nodes Snapped to Lattice: {mc_linear['levels']}")
print(f"Smart Nodes Snapped to Lattice:  {mc_smart['levels']}")

In [ ]:
print("--- RUNNING TOPOLOGICAL INTELLIGENCE SENSITIVITY ---")

# Evaluate Naive Linear Grid
print("\n[ Evaluating Naive Linear Grid... ]")
report_linear = benchmarker_linear.run_leave_one_out(eval_factory)
avg_linear = report_linear.summary.loc['Average']

# Evaluate Quantile Smart Grid
print("\n[ Evaluating Quantile Smart Grid... ]")
report_smart = benchmarker_smart.run_leave_one_out(eval_factory)
avg_smart = report_smart.summary.loc['Average']

# Combine results
df_results = pd.DataFrame({
    'Naive Linear Grid': avg_linear,
    'Quantile Smart Grid': avg_smart
}).T

In [ ]:
print("\n--- GRID TOPOLOGY: SENSITIVITY SUMMARY ---")
print_markdown_table(df_results)

# Create a targeted bar chart comparing Operational vs Battery vs Transient Costs
fig, ax = plt.subplots(figsize=(9, 5))

metrics = ['Operating Cost [$]', 'Switching Cost [$]', 'Battery Cost [$]', 'Transient Cost [$]']
linear_vals = [avg_linear[m] for m in metrics]
smart_vals = [avg_smart[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, linear_vals, width, label='Naive Linear Grid', color='lightcoral')
ax.bar(x + width/2, smart_vals, width, label='Quantile Smart Grid', color='seagreen')

ax.set_ylabel('Cost [$]', fontsize=12)
ax.set_title('Cost Breakdown: Linear vs. Smart Topology (Iso-Complexity)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([m.replace(" [$]", "") for m in metrics])
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.savefig('figures/topology_cost_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n--- PERFORMANCE VALIDATION ---")
print(f"Total Offline Compute Time (Linear): {avg_linear['Offline Compute Time [s]']:.2f} s")
print(f"Total Offline Compute Time (Smart):  {avg_smart['Offline Compute Time [s]']:.2f} s")
print(f"-> Time Difference: {abs(avg_linear['Offline Compute Time [s]'] - avg_smart['Offline Compute Time [s]']):.2f} s")
print(f"-> Cost Savings via Smart Grid:      {avg_linear['Total Cost [$]'] - avg_smart['Total Cost [$]']:+.2f} $")